In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

In [0]:
# 1. Creación del DataFrame 
ruta_csv = "/Volumes/reige_fit/default/reige/reige_fit.csv"
df_ventas = spark.read.option("header", True).option("inferSchema", True).csv(ruta_csv)

print(f"Registros iniciales leídos: {df_ventas.count()}")

Registros iniciales leídos: 1011


In [0]:
# 2. Limpieza de datos: Tipado y columnas calculadas
df_limpio = (df_ventas
    .withColumn("fecha", F.to_date("fecha", "yyyy-MM-dd"))
    .withColumn("cantidad", F.col("cantidad").cast(IntegerType()))
    .withColumn("precio", F.col("precio").cast(DoubleType()))
    .withColumn("descuento", F.col("descuento").cast(DoubleType()))
    .withColumn("subtotal", F.col("cantidad") * F.col("precio"))
    .withColumn("total_venta", F.col("subtotal") - F.col("descuento"))
)

In [0]:
# 3. Eliminación de duplicados
cant_antes = df_limpio.count()
df_limpio = df_limpio.dropDuplicates(["idVenta"])
print(f"Duplicados eliminados: {cant_antes - df_limpio.count()}")


Duplicados eliminados: 10


In [0]:
# 4. Filtrado de datos inválidos
df_limpio = df_limpio.filter((F.col("cantidad") > 0) & (F.col("total_venta") >= 0))

print(f"Registros limpios finales: {df_limpio.count()}")

Registros limpios finales: 1000


In [0]:
# 5. Guardar como tabla para que Genie (IA) y los dashboards puedan usarla
df_limpio.write.mode("overwrite").saveAsTable("ventas_limpias_reigifit")
print("Tabla 'ventas_limpias_reigifit' creada exitosamente.")

Tabla 'ventas_limpias_reigifit' creada exitosamente.


In [0]:
# Leemos la tabla que creamos en la celda anterior
df_kpi = spark.table("ventas_limpias_reigifit")

In [0]:
# INDICADOR 1: Evolución mensual de ventas
df_kpi_mes = (df_kpi
    .withColumn("mes", F.date_format("fecha", "yyyy-MM"))
    .groupBy("mes")
    .agg(
        F.sum("total_venta").alias("ventas_mensuales"),
        F.count("idVenta").alias("transacciones"),
        F.avg("total_venta").alias("ticket_promedio")
    ).orderBy("mes")
)
df_kpi_mes.write.mode("overwrite").saveAsTable("kpi_ventas_mes")

In [0]:
# INDICADOR 2: Top Productos más vendidos
df_kpi_productos = (df_kpi
    .groupBy("nombre_producto", "categoria")
    .agg(
        F.sum("cantidad").alias("unidades_vendidas"),
        F.sum("total_venta").alias("ingreso_generado")
    ).orderBy(F.desc("ingreso_generado"))
)
df_kpi_productos.write.mode("overwrite").saveAsTable("kpi_top_productos")


In [0]:
# INDICADOR 3: Clientes Recurrentes
df_kpi_clientes = (df_kpi
    .groupBy("pk_cliente", "nombre_cliente")
    .agg(
        F.count("idVenta").alias("frecuencia_compras"),
        F.sum("total_venta").alias("monto_total_gastado")
    )
    .withColumn("tipo_cliente", 
        F.when(F.col("frecuencia_compras") >= 3, "Recurrente").otherwise("Ocasional"))
    .orderBy(F.desc("monto_total_gastado"))
)
df_kpi_clientes.write.mode("overwrite").saveAsTable("kpi_clientes")

In [0]:
# INDICADOR 4: Ventas por Canal
df_kpi_canal = (df_kpi
    .groupBy("canal")
    .agg(
        F.sum("total_venta").alias("ingreso_total"),
        F.count("idVenta").alias("num_pedidos")
    )
    .withColumn("nombre_canal", 
        F.when(F.col("canal") == 1, "WhatsApp")
         .when(F.col("canal") == 2, "Instagram")
         .when(F.col("canal") == 3, "Tienda Web")
         .otherwise("Otro"))
)
df_kpi_canal.write.mode("overwrite").saveAsTable("kpi_ventas_canal")

print("Todas las tablas de indicadores (KPIs) han sido creadas.")

Todas las tablas de indicadores (KPIs) han sido creadas.


In [0]:
# Visualización rápida de la evolución de ventas
display(spark.table("kpi_ventas_mes"))

mes,ventas_mensuales,transacciones,ticket_promedio
2025-01,455.0,2,227.5
2025-02,105.0,2,52.5
2025-03,2305.0,16,144.0625
2025-04,1525.0,15,101.66666666666667
2025-05,5745.0,35,164.14285714285714
2025-06,2890.0,25,115.6
2025-07,5050.0,29,174.13793103448276
2025-08,8165.0,35,233.28571428571428
2025-09,5175.0,39,132.69230769230768
2025-10,8850.0,37,239.1891891891892


Databricks visualization. Run in Databricks to view.

In [0]:
# Visualización rápida del Top de Productos
display(spark.table("kpi_top_productos").limit(10))

nombre_producto,categoria,unidades_vendidas,ingreso_generado
Judogi Competición FUJI,Judogis,115,48065.0
Judogi Entrenamiento FUJI,Judogis,211,37390.0
Cinturón FUJI,Cinturones,593,18935.0
Rashguard FUJI,Ropa deportiva,197,16025.0
Judogi Infantil FUJI,Judogis,100,13585.0
Short de entrenamiento FUJI,Ropa deportiva,138,9265.0
Bolso deportivo FUJI,Accesorios,45,7110.0
Rodillera FUJI,Protección,116,6065.0
Protector bucal,Protección,68,1455.0


Databricks visualization. Run in Databricks to view.

In [0]:
#EXP-BD-01: Evaluación de Calidad y Duplicados
from pyspark.sql import functions as F

In [0]:
# 1. Datos crudos
df_crudo = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/reige_fit/default/reige/reige_fit.csv")
total_crudo = df_crudo.count()

In [0]:
# 2. Aplicar limpieza (simulando lo que ya hiciste)
df_limpio = df_crudo.dropDuplicates(["idVenta"]).filter(F.col("idVenta").isNotNull())
total_limpio = df_limpio.count()

print(f"Registros crudos originales: {total_crudo}")
print(f"Registros válidos después de limpieza: {total_limpio}")
print(f"Porcentaje de datos basura/duplicados eliminados: {((total_crudo - total_limpio) / total_crudo) * 100:.2f}%")

Registros crudos originales: 1011
Registros válidos después de limpieza: 1000
Porcentaje de datos basura/duplicados eliminados: 1.09%


In [0]:
#EXP-02: Ventas por la Tienda Web
# Leemos la tabla limpia guardada
df_tabla = spark.table("ventas_limpias_reigifit")

In [0]:
# Filtramos el canal 3 (Tienda Web)
df_web = df_tabla.filter("canal = 3")

print(f"Total de ventas realizadas por la Tienda Web: {df_web.count()}")
print(f"Ingreso total por la Tienda Web: S/ {df_web.agg({'total_venta': 'sum'}).collect()[0][0]:.2f}")

Total de ventas realizadas por la Tienda Web: 153
Ingreso total por la Tienda Web: S/ 27305.00


In [0]:
#EXP-BD-03: Talla más vendida
df_tabla = spark.table("ventas_limpias_reigifit")

In [0]:
# Agrupar por talla y sumar la cantidad vendida
df_tallas = df_tabla.groupBy("talla").sum("cantidad").orderBy("sum(cantidad)", ascending=False)

In [0]:
# Mostrar el top 5 de tallas
df_tallas.show(5)

+-----+-------------+
|talla|sum(cantidad)|
+-----+-------------+
|Única|          706|
|    S|          145|
|    L|          137|
|    M|          118|
|    2|           99|
+-----+-------------+
only showing top 5 rows


In [0]:
 #EXP-BD-04: Ticket Promedio
 from pyspark.sql import functions as F
 df_tabla = spark.table("ventas_limpias_reigifit")


In [0]:
# Calcular el promedio de la columna total_venta
ticket_promedio = df_tabla.agg(F.avg("total_venta")).collect()[0][0]

print(f"El ticket promedio por venta es de: S/ {ticket_promedio:.2f}")

El ticket promedio por venta es de: S/ 157.90


In [0]:
# EXP-05:  Preferencia de Métodos de Pago
from pyspark.sql import functions as F

df_tabla = spark.table("ventas_limpias_reigifit")

# Agrupamos por método de pago y traducimos los códigos a nombres
df_pagos = (df_tabla
    .groupBy("metodoPago")
    .count()
    .withColumn("nombre_metodo",
        F.when(F.col("metodoPago") == 1, "Yape")
         .when(F.col("metodoPago") == 2, "Plin")
         .when(F.col("metodoPago") == 3, "Transferencia")
         .when(F.col("metodoPago") == 4, "Efectivo")
         .when(F.col("metodoPago") == 5, "Tarjeta")
         .otherwise("Otro"))
    .orderBy(F.desc("count"))
)

# Mostramos el resultado
df_pagos.show()

+----------+-----+-------------+
|metodoPago|count|nombre_metodo|
+----------+-----+-------------+
|         1|  382|         Yape|
|         5|  267|      Tarjeta|
|         2|  123|         Plin|
|         3|  118|Transferencia|
|         4|  110|     Efectivo|
+----------+-----+-------------+

